# Experiment runner: stretch goals (GATv2 and low-homophily graphs)
Runs the proposal's stretch experiments on a Colab GPU (T4), using the same code as the main runner
(<https://github.com/YashDolorase/cs545-gat>).

| Experiment | Setup | Runs |
|---|---|---|
| GATv2 benchmark | GATv2 on Cora and CiteSeer | 2 datasets × 10 seeds |
| GATv2 under noise | GATv2 on Cora with 0 / 10 / 25 / 50 / 100 % random edges | 5 noise levels × 10 seeds |
| Low homophily | MLP, GCN, GraphSAGE, GAT, GATv2 on Texas and Chameleon | 5 models × 2 datasets × 10 splits |

**GATv2** (Brody et al., 2022) applies the nonlinearity before the attention vector:
$e_{ij} = \mathbf a^	op \mathrm{LeakyReLU}(W[h_i \Vert h_j])$. GAT's attention is *static*: every node ranks its
neighbours in the same order. GATv2's attention is *dynamic*: the ranking can depend on the node doing the attending.

On **low-homophily** graphs, most edges join nodes of different classes. Uniform neighbour averaging mixes in
features from the wrong class, so these graphs are where learned attention should help most.

**Reproducibility.** Seeds fix the initialisation, dropout and noisy graphs, but some GPU scatter operations are not deterministic, so a rerun can differ slightly from the committed `results/`. In our rerun, means moved by at most 0.5 points and no conclusion changed.

## Setup

In [1]:
import os, torch

# Clone the project repository and install PyTorch Geometric (Colab ships PyTorch but not PyG)
PROJECT = "/content/project"
if not os.path.exists(PROJECT):
    !git clone -q https://github.com/YashDolorase/cs545-gat.git {PROJECT}
os.chdir(PROJECT)
!pip -q install torch_geometric

import torch_geometric
print(f"torch {torch.__version__} | PyG {torch_geometric.__version__} | "
      f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


torch 2.14.1+cu130 | PyG 2.8.1 | device: Tesla T4


In [2]:
# Results directory: starts empty, so every run trains from scratch (the runner skips runs already in its CSV).
# Google Drive is used when available, so the results survive a disconnect.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/CS545_GAT/fresh_stretch"
except Exception:
    OUT = "/content/fresh_stretch"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT, "| existing files:", os.listdir(OUT) or "none")

Mounted at /content/drive
results -> /content/drive/MyDrive/CS545_GAT/fresh_stretch | existing files: none


## The datasets: how homophilous are they?
Edge homophily is the fraction of edges that join two nodes of the same class.
Texas (WebKB) and Chameleon (Wikipedia) come with 10 fixed train/val/test splits. Run *i* uses split *i*.

In [3]:
import pandas as pd
from src.data import load_dataset, describe, select_split

# Load all four graphs and print their size, number of classes and edge homophily.
# Texas and Chameleon store 10 splits; select_split keeps split 0 so train/val/test counts are for a single split.
stats = [describe(name, select_split(load_dataset(name, "data")[1], 0)) for name in ["Cora", "CiteSeer", "Texas", "Chameleon"]]
display(pd.DataFrame(stats).set_index("dataset"))

Processing...
Done!
Processing...
Done!
Processing...
Done!
Processing...
Done!


,nodes,edges (undirected),features,classes,train/val/test,mean degree,isolated nodes,edge homophily
dataset,,,,,,,,
Cora,2708,5278,1433,7,140/500/1000,3.90,0,0.810
CiteSeer,3327,4552,3703,6,120/500/1000,2.74,48,0.736
Texas,183,287,1703,5,87/59/37,3.14,0,0.061
Chameleon,2277,31396,2325,5,1092/729/456,27.58,0,0.230


## Low-homophily benchmark: Texas and Chameleon
All five models, each with its paper hyperparameters (as in E1), on each of the 10 splits. Results → `e1_benchmark.csv`.

In [4]:
os.chdir(PROJECT)  # Colab resets the working directory after a reconnect
!python -W ignore scripts/run_experiments.py --exp e1 --models MLP GCN SAGE GAT GATv2 --datasets Texas Chameleon --out {OUT}

device: cuda | torch 2.14.1+cu130
[E1] Texas     MLP    seed 0: test 0.784 (best ep 353, 1.7 ms/ep)
[E1] Texas     GCN    seed 0: test 0.649 (best ep 446, 3.8 ms/ep)
[E1] Texas     SAGE   seed 0: test 0.811 (best ep 273, 3.1 ms/ep)
[E1] Texas     GAT    seed 0: test 0.622 (best ep 343, 7.1 ms/ep)
[E1] Texas     GATv2  seed 0: test 0.622 (best ep 237, 5.6 ms/ep)
[E1] Texas     MLP    seed 1: test 0.865 (best ep 512, 1.8 ms/ep)
[E1] Texas     GCN    seed 1: test 0.595 (best ep 421, 3.4 ms/ep)
[E1] Texas     SAGE   seed 1: test 0.838 (best ep 525, 3.2 ms/ep)
[E1] Texas     GAT    seed 1: test 0.541 (best ep 172, 6.4 ms/ep)
[E1] Texas     GATv2  seed 1: test 0.568 (best ep 587, 5.3 ms/ep)
[E1] Texas     MLP    seed 2: test 0.730 (best ep 438, 1.7 ms/ep)
[E1] Texas     GCN    seed 2: test 0.568 (best ep 343, 4.0 ms/ep)
[E1] Texas     SAGE   seed 2: test 0.730 (best ep 430, 3.0 ms/ep)
[E1] Texas     GAT    seed 2: test 0.514 (best ep 423, 5.8 ms/ep)
[E1] Texas     GATv2  seed 2: test 0.486 (

## GATv2 on Cora and CiteSeer
Same protocol and seeds as E1, so these runs can be paired with the GAT, GCN and GraphSAGE runs from the main runner.
Results → `e1_benchmark.csv`.

In [5]:
os.chdir(PROJECT)
!python -W ignore scripts/run_experiments.py --exp e1 --models GATv2 --datasets Cora CiteSeer --out {OUT}

device: cuda | torch 2.14.1+cu130
[E1] Cora      GATv2  seed 0: test 0.826 (best ep 41, 5.9 ms/ep)
[E1] Cora      GATv2  seed 1: test 0.825 (best ep 123, 7.4 ms/ep)
[E1] Cora      GATv2  seed 2: test 0.831 (best ep 167, 6.5 ms/ep)
[E1] Cora      GATv2  seed 3: test 0.823 (best ep 57, 5.9 ms/ep)
[E1] Cora      GATv2  seed 4: test 0.813 (best ep 143, 5.9 ms/ep)
[E1] Cora      GATv2  seed 5: test 0.829 (best ep 325, 6.0 ms/ep)
[E1] Cora      GATv2  seed 6: test 0.810 (best ep 40, 6.8 ms/ep)
[E1] Cora      GATv2  seed 7: test 0.824 (best ep 177, 7.1 ms/ep)
[E1] Cora      GATv2  seed 8: test 0.812 (best ep 64, 6.1 ms/ep)
[E1] Cora      GATv2  seed 9: test 0.831 (best ep 153, 5.9 ms/ep)
[E1] CiteSeer  GATv2  seed 0: test 0.695 (best ep 182, 7.0 ms/ep)
[E1] CiteSeer  GATv2  seed 1: test 0.725 (best ep 83, 7.3 ms/ep)
[E1] CiteSeer  GATv2  seed 2: test 0.714 (best ep 157, 8.7 ms/ep)
[E1] CiteSeer  GATv2  seed 3: test 0.713 (best ep 78, 7.0 ms/ep)
[E1] CiteSeer  GATv2  seed 4: test 0.728 (best e

## GATv2 under noise
Same noisy Cora graphs as E3: each seed adds the same random edges as in the main runner.
Results, including GATv2's attention on injected and original edges, → `e3_noise.csv`.

In [6]:
os.chdir(PROJECT)
!python -W ignore scripts/run_experiments.py --exp e3 --models GATv2 --out {OUT}

device: cuda | torch 2.14.1+cu130
[E3] Cora noise 0.00 GATv2  seed 0: test 0.826
[E3] Cora noise 0.00 GATv2  seed 1: test 0.825
[E3] Cora noise 0.00 GATv2  seed 2: test 0.831
[E3] Cora noise 0.00 GATv2  seed 3: test 0.823
[E3] Cora noise 0.00 GATv2  seed 4: test 0.812
[E3] Cora noise 0.00 GATv2  seed 5: test 0.829
[E3] Cora noise 0.00 GATv2  seed 6: test 0.810
[E3] Cora noise 0.00 GATv2  seed 7: test 0.832
[E3] Cora noise 0.00 GATv2  seed 8: test 0.812
[E3] Cora noise 0.00 GATv2  seed 9: test 0.832
[E3] Cora noise 0.10 GATv2  seed 0: test 0.811
[E3] Cora noise 0.10 GATv2  seed 1: test 0.794
[E3] Cora noise 0.10 GATv2  seed 2: test 0.822
[E3] Cora noise 0.10 GATv2  seed 3: test 0.809
[E3] Cora noise 0.10 GATv2  seed 4: test 0.795
[E3] Cora noise 0.10 GATv2  seed 5: test 0.787
[E3] Cora noise 0.10 GATv2  seed 6: test 0.789
[E3] Cora noise 0.10 GATv2  seed 7: test 0.795
[E3] Cora noise 0.10 GATv2  seed 8: test 0.791
[E3] Cora noise 0.10 GATv2  seed 9: test 0.780
[E3] Cora noise 0.25 GATv2

## Results summary

In [7]:
# Stretch results; the main runner's results are added to the same tables when they are in the sibling folder
MAIN = os.path.join(os.path.dirname(OUT), "fresh_run")
def load(name):
    df = pd.read_csv(f"{OUT}/{name}")
    if os.path.exists(f"{MAIN}/{name}"):
        df = pd.concat([df, pd.read_csv(f"{MAIN}/{name}")], ignore_index=True)
    return df

# Test accuracy (%) as mean ± std over seeds/splits
e1 = load("e1_benchmark.csv")
s = e1.groupby(["dataset", "model"])["test_acc"].agg(["mean", "std"])
s = (100 * s["mean"]).round(1).astype(str) + " ± " + (100 * s["std"]).round(1).astype(str)
display(s.unstack("dataset")[[d for d in ["Cora", "CiteSeer", "Texas", "Chameleon"] if d in e1.dataset.unique()]])

# Noisy Cora: mean test accuracy (%) per noise level (rows) and model (columns)
e3 = load("e3_noise.csv")
display((100 * e3.pivot_table(index="noise", columns="model", values="test_acc", aggfunc="mean")).round(1))

# Attention on injected vs original edges relative to uniform (1.0 = what uniform weights would give)
att = ["L1_att_original", "L1_att_injected", "L2_att_original", "L2_att_injected"]
display(e3[e3.model.isin(["GAT", "GATv2"]) & (e3.noise > 0)].groupby(["model", "noise"])[att].mean().round(3))

dataset,Cora,CiteSeer,Texas,Chameleon
model,,,,
GAT,NaN,NaN,58.4 ± 4.1,57.5 ± 2.0
GATv2,82.2 ± 0.8,72.0 ± 1.3,58.9 ± 5.1,58.2 ± 2.4
GCN,NaN,NaN,62.2 ± 4.0,55.5 ± 1.5
MLP,NaN,NaN,80.3 ± 5.6,47.4 ± 1.6
SAGE,NaN,NaN,84.1 ± 5.6,62.0 ± 1.9


model,GATv2
noise,
0.00,82.3
0.10,79.7
0.25,77.3
0.50,72.6
1.00,67.4


L1_att_original  L1_att_injected  L2_att_original  \
model noise                                                      
GATv2 0.10             1.001            0.998            0.999   
      0.25             1.001            0.999            1.002   
      0.50             1.001            0.999            1.010   
      1.00             1.001            1.000            1.016   

             L2_att_injected  
model noise                   
GATv2 0.10             0.992  
      0.25             0.988  
      0.50             0.974  
      1.00             0.980

In [8]:
# Package the results directory for download
!cd {os.path.dirname(OUT)} && zip -qr /content/fresh_stretch.zip {os.path.basename(OUT)}
from google.colab import files
files.download("/content/fresh_stretch.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>